# OCR des 17 PDFs scannés — House Q1 2025

**Contexte** : Le pipeline principal (`notebook_v1_house_2025q1.ipynb`) classe 17 PTRs dans `data_v1/pdfs/non_lisibles/` car `pdfplumber` n'y trouve aucun texte. Ces PDFs sont des scans de formulaires papier (images JPEG/PNG encapsulées dans un PDF, sans couche texte).

**Approche** : Claude Vision API (claude-sonnet-4-6) — envoi de chaque page comme image, extraction structurée des transactions en JSON.

**Output** :
- Cache par PDF : `data_v1/ocr_cache/{doc_id}.json`
- Table OCR : `data_v1/tables/06b_house_2025q1_ocr_transactions.csv`
- Table combinée : `data_v1/tables/06_house_2025q1_transactions_FINAL.csv`

**Prérequis** : Ajouter `ANTHROPIC_API_KEY=sk-ant-...` dans le fichier `.env` (clé API Anthropic, distincte de la souscription claude.ai — obtenir sur console.anthropic.com).

> **Étape 2/2 du pipeline House** — à lancer **après** `notebook_v1_house_2025q1.ipynb` (qui produit la table électronique `06_…transactions.csv`). Ce notebook ajoute l'OCR des 17 PTR scannés, fusionne en **`06_…transactions_FINAL.csv`**, valide vs Quiver et génère **`house_2025q1_FINAL.xlsx`**. Voir `README.md`.

## 0. Setup & imports

In [1]:
import base64
import hashlib
import json
import os
import re
import time
from pathlib import Path

import anthropic
import pandas as pd
import pymupdf
from dotenv import load_dotenv

# --- Ancrage du dossier projet (indépendant du répertoire de lancement) ---
# On remonte depuis le CWD jusqu'au dossier des notebooks (marqueur stable) :
# le notebook tourne pareil qu'on le lance depuis 2025_test/ ou un sous-dossier.
BASE_DIR = Path.cwd()
for _p in [BASE_DIR, *BASE_DIR.parents]:
    if (_p / "notebook_v1_house_2025q1_ocr.ipynb").exists():
        BASE_DIR = _p
        break
assert (BASE_DIR / "notebook_v1_house_2025q1_ocr.ipynb").exists(), (
    "Lance Jupyter depuis 0 HOUSE/2025_test/ (dossier des notebooks)."
)

# --- Clés API : .env LOCAL au dossier (autonome) ; repli walk-up si absent ---
load_dotenv(BASE_DIR / ".env")
if not os.getenv("ANTHROPIC_API_KEY"):
    load_dotenv()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
if not ANTHROPIC_API_KEY:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY manquante dans .env\n"
        "Ajoutez : ANTHROPIC_API_KEY=sk-ant-...\n"
        "Obtenir sur : https://console.anthropic.com/settings/keys"
    )

PDF_DIR     = BASE_DIR / "data_v1/pdfs/non_lisibles"
TABLE_DIR   = BASE_DIR / "data_v1/tables"
OCR_CACHE   = BASE_DIR / "data_v1/ocr_cache"
OCR_CACHE.mkdir(parents=True, exist_ok=True)

MODEL            = "claude-sonnet-4-6"
DPI              = 200
MAX_IMG_PER_CALL = 3       # pages par appel — petit batch = meilleure exhaustivité sur tables denses
MAX_TOKENS       = 16_000  # marge confortable pour la sortie structurée d'un batch dense

print(f"anthropic {anthropic.__version__} | pymupdf {pymupdf.__version__}")
print(f"BASE_DIR   : {BASE_DIR.name}/ (résolu localement)")
print(f"PDF_DIR    : {PDF_DIR.relative_to(BASE_DIR)}")
print(f"OCR_CACHE  : {OCR_CACHE.relative_to(BASE_DIR)}")


anthropic 0.112.0 | pymupdf 1.27.2.3
BASE_DIR   : 2025_test/ (résolu localement)
PDF_DIR    : data_v1/pdfs/non_lisibles
OCR_CACHE  : data_v1/ocr_cache


## 1. Inventaire des 17 PDFs scannés

In [2]:

# 04_download_manifest.csv : doc_id | url | status | has_text
# 03_ptr_index.csv         : doc_id | declarant_name | state_district | disclosure_date | url_pdf | ...
manifest  = pd.read_csv(TABLE_DIR / "04_download_manifest.csv", dtype={"doc_id": str})
ptr_index = pd.read_csv(TABLE_DIR / "03_ptr_index.csv",         dtype={"doc_id": str})

scanned_ids = manifest.loc[~manifest["has_text"], "doc_id"]
scanned = ptr_index[ptr_index["doc_id"].isin(scanned_ids)].copy().reset_index(drop=True)

# Compter les pages de chaque PDF
page_counts = {}
for doc_id in scanned["doc_id"]:
    p = PDF_DIR / f"{doc_id}.pdf"
    page_counts[doc_id] = len(pymupdf.open(p)) if p.exists() else 0

scanned["n_pages"] = scanned["doc_id"].map(page_counts)
scanned["pdf_exists"] = scanned["doc_id"].apply(lambda d: (PDF_DIR / f"{d}.pdf").exists())

total_pages = scanned["n_pages"].sum()
cost_est = total_pages * 0.006  # ~$0.006/image pour claude-sonnet-4-6

print(f"PDFs scannés : {len(scanned)} | Pages totales : {total_pages} | Coût estimé : ${cost_est:.2f}\n")
print(scanned[["doc_id", "declarant_name", "state_district", "disclosure_date", "n_pages", "pdf_exists"]].to_string(index=False))


PDFs scannés : 17 | Pages totales : 90 | Coût estimé : $0.54

 doc_id                 declarant_name state_district disclosure_date  n_pages  pdf_exists
8220747               Gus M. Bilirakis           FL12      2025-02-06        2        True
8220753 Charles J. "Chuck" Fleischmann           TN03      2025-02-10        4        True
8220796               Vicente Gonzalez           TX34      2025-03-13        1        True
8220809               Vicente Gonzalez           TX34      2025-03-31        1        True
8220750                   Rohit Khanna           CA17      2025-02-06       34        True
8220783                   Rohit Khanna           CA17      2025-03-06       28        True
8220754              Michael T. McCaul           TX10      2025-02-11        4        True
8220755           Harold Dallas Rogers           KY05      2025-02-12        1        True
8220782           Harold Dallas Rogers           KY05      2025-03-06        1        True
8220731                Keith

## 2. Constantes : fourchettes de montant et prompt OCR

Les formulaires PTR utilisent des cases à cocher A–K pour les montants. Le prompt instruit Claude sur la structure exacte du formulaire.

In [3]:
# Correspondance lettre → (fourchette affichée, midpoint)
AMOUNT_MAP = {
    "A": ("$1,001 - $15,000",             8_000.0),
    "B": ("$15,001 - $50,000",            32_500.0),
    "C": ("$50,001 - $100,000",           75_000.0),
    "D": ("$100,001 - $250,000",         175_000.0),
    "E": ("$250,001 - $500,000",         375_000.0),
    "F": ("$500,001 - $1,000,000",       750_000.0),
    "G": ("$1,000,001 - $5,000,000",   3_000_000.0),
    "H": ("$5,000,001 - $25,000,000",  15_000_000.0),
    "I": ("$25,000,001 - $50,000,000", 37_500_000.0),
    "J": ("Over $50,000,000",           75_000_000.0),
    "K": ("SP/DC over $1,000,000",       1_000_001.0),
}

OCR_PROMPT = """\
Tu lis les pages scannées d'un formulaire PTR (Periodic Transaction Report — US House of
Representatives) déposé par {member_name}. Reporte TOUTES les transactions financières en
appelant l'outil record_transactions.

STRUCTURE DU FORMULAIRE :
  Colonne propriétaire (gauche, étroite) : DC = Dependent Child | JT = Joint | SP = Spouse | vide = Self
  FULL ASSET NAME : nom complet de l'actif tel qu'écrit (n'invente pas de ticker)
  TYPE OF TRANSACTION (cases à cocher) : Purchase | Sale | Partial Sale | Exchange
  DATE OF TRANSACTION et DATE NOTIFIED OF TRANSACTION : format MM/DD/YY (deux dates distinctes par ligne)
  AMOUNT OF TRANSACTION (cases A–K, une seule cochée par ligne) :
    A=$1,001–15k  B=15k–50k  C=50k–100k  D=100k–250k  E=250k–500k  F=500k–1M
    G=1M–5M  H=5M–25M  I=25M–50M  J=>50M  K=SP/DC >1M

RÈGLES CRITIQUES :
  1. Le formulaire peut être scanné à 90° ou 180° — lis-le dans son orientation correcte.
  2. LIGNE D'EXEMPLE PRÉ-IMPRIMÉE — À IGNORER ABSOLUMENT. Chaque formulaire vierge contient une
     ligne modèle imprimée « Example: Mega Corp. Common Stock » (souvent accompagnée de la mention
     « Provide full name, not ticker symbol »). Ce N'EST JAMAIS une transaction : ne la reporte pas.
  3. « Nothing to report for <mois> » dans la colonne FULL ASSET NAME → aucune transaction sur cette page.
  4. Ignore les pages de couverture / certification sans table de transactions.
  5. Reporte CHAQUE ligne dont une case TYPE est cochée, Y COMPRIS des lignes strictement identiques
     répétées (même actif, même date, même montant, même propriétaire) : ce sont des transactions
     réelles distinctes, jamais des doublons à fusionner.
  6. Convertis MM/DD/YY → YYYY-MM-DD (siècle 2000+). Lis transaction_date ET notification_date
     ligne par ligne ; ne recopie pas une date unique sur toutes les lignes.
  7. amount_code = UNIQUEMENT la lettre de la case cochée (A–K). Si illisible, omets le champ.
"""

# Schéma structuré FORCÉ (tool use) — supprime tout parsing JSON fragile et valide les enums à la source
TXN_TOOL = {
    "name": "record_transactions",
    "description": (
        "Enregistre la liste complète des transactions lues sur les pages fournies. "
        "Tableau vide si aucune transaction (ex: page 'Nothing to report' ou page de couverture)."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "transactions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "asset_description":  {"type": "string", "description": "nom complet de l'actif tel qu'écrit sur le formulaire"},
                        "transaction_type":   {"type": "string", "enum": ["Purchase", "Sale", "Partial Sale", "Exchange"]},
                        "transaction_date":   {"type": ["string", "null"], "description": "YYYY-MM-DD ou null si illisible"},
                        "notification_date":  {"type": ["string", "null"], "description": "YYYY-MM-DD ou null si illisible"},
                        "amount_code":        {"type": "string", "enum": ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K"]},
                        "owner":              {"type": "string", "enum": ["Self", "Spouse", "Joint", "Dependent Child"]},
                    },
                    "required": ["asset_description", "transaction_type", "owner"],
                    "additionalProperties": False,
                },
            }
        },
        "required": ["transactions"],
    },
}

# Empreinte prompt+schéma : invalide automatiquement le cache si l'un des deux change
PROMPT_SHA = hashlib.sha256((OCR_PROMPT + json.dumps(TXN_TOOL, sort_keys=True)).encode()).hexdigest()[:12]
print("Constantes : AMOUNT_MAP (%d codes) | OCR_PROMPT (%d chars) | tool=record_transactions | prompt_sha=%s"
      % (len(AMOUNT_MAP), len(OCR_PROMPT), PROMPT_SHA))

Constantes : AMOUNT_MAP (11 codes) | OCR_PROMPT (1934 chars) | tool=record_transactions | prompt_sha=608a8597d2d4


## 3. Fonctions d'extraction (PDF → images → Claude Vision → JSON)

In [4]:
def pdf_to_b64_images(pdf_path: Path, dpi: int = DPI) -> list:
    """Convertit chaque page du PDF en PNG base64 pour l'API Anthropic."""
    doc = pymupdf.open(pdf_path)
    images = []
    for page in doc:
        pix = page.get_pixmap(dpi=dpi)
        images.append(base64.b64encode(pix.tobytes("png")).decode())
    return images


class OcrError(Exception):
    """Échec d'extraction d'un batch (signalé, jamais avalé silencieusement)."""


def _call_vision_tool(client, images_b64: list, member_name: str, max_retries: int = 4) -> list:
    """Un appel API avec tool use FORCÉ + retry/backoff. Retourne la liste de transactions.

    Le tool_choice force le modèle à appeler record_transactions → plus de JSON-dans-le-texte,
    plus de troncature à réparer, enums validés à la source. Lève OcrError si l'appel échoue
    après retries ou si aucun bloc tool_use n'est renvoyé.
    """
    content = [
        {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": b64}}
        for b64 in images_b64
    ]
    content.append({"type": "text", "text": OCR_PROMPT.format(member_name=member_name)})

    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                tools=[TXN_TOOL],
                tool_choice={"type": "tool", "name": "record_transactions"},
                messages=[{"role": "user", "content": content}],
            )
            for block in resp.content:
                if getattr(block, "type", None) == "tool_use" and block.name == "record_transactions":
                    return block.input.get("transactions", [])
            raise OcrError(f"aucun bloc tool_use (stop_reason={resp.stop_reason})")
        except (anthropic.RateLimitError, anthropic.APIStatusError, anthropic.APIConnectionError) as e:
            last_err = e
            status = getattr(e, "status_code", None)
            retriable = isinstance(e, (anthropic.RateLimitError, anthropic.APIConnectionError)) or status in (500, 502, 503, 529)
            if retriable and attempt < max_retries - 1:
                time.sleep(min(2 ** attempt, 20))
                continue
            raise OcrError(f"appel API échoué : {type(e).__name__} {status} {e}") from e
    raise OcrError(f"échec après {max_retries} tentatives : {last_err}")


def extract_from_pdf(pdf_path: Path, doc_id: str, member_name: str, force: bool = False) -> tuple:
    """Extraction complète avec cache disque VERSIONNÉ et batching. Retourne (transactions, cache_obj).

    Le cache est un objet auditable {model, prompt_sha, batches[…], transactions}. Il est réutilisé
    seulement si model + prompt_sha correspondent (sinon ré-extraction). Le statut distingue
    'nothing_to_report' (vide légitime) de 'partial_error' (au moins un batch a échoué).
    """
    cache_file = OCR_CACHE / f"{doc_id}.json"
    if cache_file.exists() and not force:
        try:
            cached = json.loads(cache_file.read_text())
        except json.JSONDecodeError:
            cached = None
        if isinstance(cached, dict) and cached.get("prompt_sha") == PROMPT_SHA and cached.get("model") == MODEL:
            print(f"  [cache] {doc_id} ({len(cached['transactions'])} txns, {cached.get('status')})")
            return cached["transactions"], cached
        # ancien cache (tableau nu) ou prompt/modèle changé → on ré-extrait

    images = pdf_to_b64_images(pdf_path)
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    batches = [images[i: i + MAX_IMG_PER_CALL] for i in range(0, len(images), MAX_IMG_PER_CALL)]

    all_txns, batch_log, status = [], [], "ok"
    for idx, batch in enumerate(batches):
        multi = len(batches) > 1
        if multi:
            print(f"    batch {idx+1}/{len(batches)} ({len(batch)} p.)", end="", flush=True)
        try:
            txns = _call_vision_tool(client, batch, member_name)
            all_txns.extend(txns)
            batch_log.append({"batch": idx, "pages": len(batch), "status": "ok", "n": len(txns)})
            if multi:
                print(f" → {len(txns)} txns")
        except OcrError as e:
            status = "partial_error"
            batch_log.append({"batch": idx, "pages": len(batch), "status": "error", "error": str(e)[:200]})
            if multi:
                print(f" → ⚠ ERREUR : {e}")
        if idx < len(batches) - 1:
            time.sleep(0.4)

    cache_obj = {
        "doc_id": doc_id,
        "member": member_name,
        "model": MODEL,
        "prompt_sha": PROMPT_SHA,
        "dpi": DPI,
        "max_img_per_call": MAX_IMG_PER_CALL,
        "n_pages": len(images),
        "created_utc": pd.Timestamp.utcnow().isoformat(),
        "status": "nothing_to_report" if (status == "ok" and not all_txns) else status,
        "batches": batch_log,
        "transactions": all_txns,
    }
    cache_file.write_text(json.dumps(cache_obj, ensure_ascii=False, indent=2))
    return all_txns, cache_obj


print(f"Fonctions chargées : tool use forcé (record_transactions) | retry/backoff | cache versionné (prompt_sha={PROMPT_SHA}) | max_tokens={MAX_TOKENS}")

Fonctions chargées : tool use forcé (record_transactions) | retry/backoff | cache versionné (prompt_sha=608a8597d2d4) | max_tokens=16000


## 4. Test sur un PDF représentatif avant la boucle complète

On vérifie d'abord que l'extraction fonctionne sur Rogers (le plus simple, "nothing to report") et McCaul (4 pages, many transactions).

In [5]:
# Test sur Rogers (1 page, "Nothing to report") — doit retourner [] avec status=nothing_to_report
row_test = scanned[scanned["declarant_name"].str.contains("Rogers", na=False)].iloc[0]
doc_id_test = row_test["doc_id"]
pdf_test = PDF_DIR / f"{doc_id_test}.pdf"

print(f"Test sur : {row_test['declarant_name']} ({doc_id_test}, {page_counts.get(doc_id_test, '?')} page(s))")
txns_test, cache_test = extract_from_pdf(pdf_test, doc_id_test, row_test["declarant_name"])
print(f"Résultat : {len(txns_test)} transaction(s) | status={cache_test['status']}")
print(json.dumps(txns_test, indent=2, ensure_ascii=False))

Test sur : Harold Dallas Rogers (8220755, 1 page(s))
  [cache] 8220755 (0 txns, nothing_to_report)
Résultat : 0 transaction(s) | status=nothing_to_report
[]


In [6]:
# Test sur McCaul (4 pages, transactions multiples)
row_mccaul = scanned[scanned["declarant_name"].str.contains("McCaul", na=False)].iloc[0]
doc_id_mc = row_mccaul["doc_id"]
pdf_mc = PDF_DIR / f"{doc_id_mc}.pdf"

print(f"Test sur : {row_mccaul['declarant_name']} ({doc_id_mc}, {page_counts.get(doc_id_mc, '?')} pages)")
txns_mc, cache_mc = extract_from_pdf(pdf_mc, doc_id_mc, row_mccaul["declarant_name"])
print(f"Résultat : {len(txns_mc)} transaction(s) | status={cache_mc['status']}")
for t in txns_mc[:5]:
    print(" ", t)

Test sur : Michael T. McCaul (8220754, 4 pages)
  [cache] 8220754 (82 txns, ok)
Résultat : 82 transaction(s) | status=ok
  {'asset_description': 'LIM FAMILY INVESTMENTS LP', 'transaction_type': 'Sale', 'owner': 'Spouse', 'transaction_date': '2025-01-28', 'notification_date': '2025-02-05', 'amount_code': 'G'}
  {'asset_description': 'COLUMBIA FLOATING RATE', 'transaction_type': 'Purchase', 'owner': 'Spouse', 'transaction_date': '2025-01-22', 'notification_date': '2025-02-05', 'amount_code': 'E'}
  {'asset_description': 'ALLOCATE PREMIER ACCESS FUND', 'transaction_type': 'Purchase', 'owner': 'Spouse', 'transaction_date': '2025-01-22', 'notification_date': '2025-02-05', 'amount_code': 'D'}
  {'asset_description': 'OHIO ST HIGHER ED GO BDS', 'transaction_type': 'Purchase', 'owner': 'Spouse', 'transaction_date': '2025-01-22', 'notification_date': '2025-02-05', 'amount_code': 'D'}
  {'asset_description': 'HUMANA INC', 'transaction_type': 'Purchase', 'owner': 'Spouse', 'transaction_date': '20

## 5. Boucle principale — les 17 PDFs

Chaque PDF est traité avec cache disque : si `ocr_cache/{doc_id}.json` existe déjà, il est relu sans appel API.

In [7]:
raw_results = []   # [{doc_id, declarant_name, transactions: [...]}]
ocr_failures = []  # batches en erreur — journalisés BRUYAMMENT, jamais avalés

for _, row in scanned.iterrows():
    doc_id  = row["doc_id"]
    member  = row["declarant_name"]
    n_pages = page_counts.get(doc_id, 0)
    pdf_path = PDF_DIR / f"{doc_id}.pdf"

    if not pdf_path.exists():
        print(f"⚠ PDF manquant : {doc_id} — {member}")
        ocr_failures.append({"doc_id": doc_id, "declarant_name": member, "batch": -1, "reason": "pdf_manquant"})
        continue

    print(f"→ {doc_id}  {member}  ({n_pages} p.)")
    txns, cache_obj = extract_from_pdf(pdf_path, doc_id, member)
    print(f"   ✓ {len(txns)} transaction(s)  [{cache_obj['status']}]")
    raw_results.append({"doc_id": doc_id, "declarant_name": member, "transactions": txns})
    for bl in cache_obj.get("batches", []):
        if bl.get("status") == "error":
            ocr_failures.append({"doc_id": doc_id, "declarant_name": member,
                                 "batch": bl["batch"], "reason": bl.get("error", "")})

# Journal des échecs (un fichier VIDE est le bon résultat)
fail_df = pd.DataFrame(ocr_failures, columns=["doc_id", "declarant_name", "batch", "reason"])
fail_df.to_csv(TABLE_DIR / "06c_ocr_failures.csv", index=False)

n_total = sum(len(r["transactions"]) for r in raw_results)
n_ok    = sum(1 for r in raw_results if r["transactions"])
n_empty = sum(1 for r in raw_results if not r["transactions"])
print(f"\n=== {n_total} transactions | {n_ok} PDFs avec données | {n_empty} vides | {len(ocr_failures)} échec(s) batch ===")
if ocr_failures:
    print("⚠ ÉCHECS À INVESTIGUER (06c_ocr_failures.csv) :")
    print(fail_df.to_string(index=False))
else:
    print("✓ aucun échec de batch — extraction complète")

→ 8220747  Gus M. Bilirakis  (2 p.)
  [cache] 8220747 (2 txns, ok)
   ✓ 2 transaction(s)  [ok]
→ 8220753  Charles J. "Chuck" Fleischmann  (4 p.)
  [cache] 8220753 (21 txns, ok)
   ✓ 21 transaction(s)  [ok]
→ 8220796  Vicente Gonzalez  (1 p.)
  [cache] 8220796 (3 txns, ok)
   ✓ 3 transaction(s)  [ok]
→ 8220809  Vicente Gonzalez  (1 p.)
  [cache] 8220809 (1 txns, ok)
   ✓ 1 transaction(s)  [ok]
→ 8220750  Rohit Khanna  (34 p.)
  [cache] 8220750 (589 txns, ok)
   ✓ 589 transaction(s)  [ok]
→ 8220783  Rohit Khanna  (28 p.)
  [cache] 8220783 (447 txns, ok)
   ✓ 447 transaction(s)  [ok]
→ 8220754  Michael T. McCaul  (4 p.)
  [cache] 8220754 (82 txns, ok)
   ✓ 82 transaction(s)  [ok]
→ 8220755  Harold Dallas Rogers  (1 p.)
  [cache] 8220755 (0 txns, nothing_to_report)
   ✓ 0 transaction(s)  [nothing_to_report]
→ 8220782  Harold Dallas Rogers  (1 p.)
  [cache] 8220782 (0 txns, nothing_to_report)
   ✓ 0 transaction(s)  [nothing_to_report]
→ 8220731  Keith Alan Self  (2 p.)
  [cache] 8220731 (5 

## 6. Normalisation → DataFrame au schéma du pipeline

In [8]:
OWNER_MAP = {
    "Self":             "SELF",
    "Spouse":           "Spouse",
    "Joint":            "Joint Tenancy",
    "Dependent Child":  "Dependent Child",
}

# Filet de sécurité DÉTERMINISTE : la ligne d'exemple pré-imprimée du formulaire
# (« Example: Mega Corp. Common Stock ») n'est JAMAIS un actif réel. Le prompt demande de
# l'ignorer, mais sur un formulaire quasi vide le modèle la lit parfois quand même → on la
# retire ici par règle, garantissant qu'elle n'atteint jamais la sortie.
_EXAMPLE_RE = re.compile(r"example.*mega\s*corp|mega\s*corp.*common\s*stock", re.I)


def _is_example_row(txn: dict) -> bool:
    return bool(_EXAMPLE_RE.search(str(txn.get("asset_description", ""))))


def natural_key_hash(row: dict) -> str:
    key = "|".join([
        "house",
        str(row.get("declarant_name", "")),
        str(row.get("transaction_date", "")),
        str(row.get("asset_description", "")),
        str(row.get("operation_type", "")),
        str(row.get("amount_range", "")),
        str(row.get("owner", "")),
    ])
    return hashlib.sha256(key.encode()).hexdigest()


def normalize(txn: dict, meta: dict) -> dict:
    """meta vient de la ligne ptr_index (doc_id, declarant_name, state_district, disclosure_date, url_pdf)."""
    code = (txn.get("amount_code") or "").upper()
    amount_range, amount_mid = AMOUNT_MAP.get(code, ("", None))
    owner_raw = txn.get("owner") or "Self"
    r = {
        "bioguide_id":           None,
        "declarant_name":        meta["declarant_name"],
        "chamber":               "house",
        "party":                 None,
        "state_district":        meta.get("state_district", ""),
        "committee_membership":  None,
        "committees_key_flag":   None,
        "transaction_date":      txn.get("transaction_date"),
        "disclosure_date":       txn.get("notification_date") or meta.get("disclosure_date", ""),
        "ticker":                None,
        "asset_description":     txn.get("asset_description", ""),
        "asset_type":            None,
        "operation_type":        txn.get("transaction_type", ""),
        "amount_range":          amount_range,
        "amount_midpoint":       amount_mid,
        "amount_split_flag":     False,
        "owner":                 OWNER_MAP.get(owner_raw, "SELF"),
        "doc_id":                meta["doc_id"],
        "source_url":            meta.get("url_pdf", f"https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/{meta['doc_id']}.pdf"),
        "natural_key_hash":      None,
        "provenance":            "house-pdf-ocr",
    }
    r["natural_key_hash"] = natural_key_hash(r)
    return r


meta_lookup = {row["doc_id"]: row.to_dict() for _, row in scanned.iterrows()}

rows, n_example = [], 0
for entry in raw_results:
    meta = meta_lookup.get(entry["doc_id"], {"doc_id": entry["doc_id"], "declarant_name": entry["declarant_name"]})
    for txn in entry["transactions"]:
        if _is_example_row(txn):
            n_example += 1
            continue
        rows.append(normalize(txn, meta))

df_ocr = pd.DataFrame(rows)
print(f"DataFrame OCR : {len(df_ocr)} lignes ({n_example} ligne(s)-exemple ecartee(s) par securite)")
if not df_ocr.empty:
    print(df_ocr[["declarant_name", "operation_type", "amount_range", "owner", "asset_description"]].head(20).to_string(index=False))


DataFrame OCR : 1167 lignes (1 ligne(s)-exemple ecartee(s) par securite)
                declarant_name operation_type      amount_range           owner                       asset_description
              Gus M. Bilirakis           Sale $15,001 - $50,000            SELF                    GE Vernova LLC (GEV)
              Gus M. Bilirakis           Sale                   Dependent Child                    GE Vernova LLC (GEV)
Charles J. "Chuck" Fleischmann           Sale $15,001 - $50,000   Joint Tenancy                   Amplify ETF Tr - HACK
Charles J. "Chuck" Fleischmann       Purchase  $1,001 - $15,000   Joint Tenancy        Franklin Templeton ETF TR - FLJP
Charles J. "Chuck" Fleischmann           Sale  $1,001 - $15,000   Joint Tenancy        Franklin Templeton ETF TR - FLJP
Charles J. "Chuck" Fleischmann       Purchase  $1,001 - $15,000   Joint Tenancy            Global X Funds Uranium - URA
Charles J. "Chuck" Fleischmann       Purchase  $1,001 - $15,000   Joint Tenancy        

## 7. Métriques de résultat

## 6b. Enrichissement : BioGuideID, party, committee_membership (via ref_universe)

In [9]:

if df_ocr.empty:
    print("Aucune transaction — skip enrichissement")
else:
    ref = pd.read_csv(TABLE_DIR / "01_ref_universe.csv", dtype=str)
    ref_house = ref[ref["chamber"] == "house"].copy()   # NB: valeur = "house" pas "rep"

    key = pd.read_csv(TABLE_DIR / "02_ref_house_key.csv", dtype=str)
    key_ids = set(key["bioguide_id"].dropna())

    # Map exact declarant_name → bioguide_id / party
    name_to_bio   = ref_house.set_index("declarant_name")["bioguide_id"].to_dict()
    name_to_party = ref_house.set_index("declarant_name")["party"].to_dict()

    # Surcharges manuelles pour les noms qui divergent entre ptr_index et ref_universe :
    #   ptr_index "Rohit Khanna"        ←→ ref "Ro Khanna"        (K000389)
    #   ptr_index "Harold Dallas Rogers" ←→ ref "Harold Rogers"    (R000395)
    #   ptr_index "Keith Alan Self"      ←→ ref "Keith Self"       (S001224)
    MANUAL_BIO = {
        "Rohit Khanna":         ("K000389", "Democrat"),
        "Harold Dallas Rogers": ("R000395", "Republican"),
        "Keith Alan Self":      ("S001224", "Republican"),
    }
    for name, (bio, party) in MANUAL_BIO.items():
        name_to_bio[name]   = bio
        name_to_party[name] = party

    # Committee membership par bioguide_id
    cmte = key.groupby("bioguide_id")["committee_category"].apply(lambda s: "; ".join(s.dropna())).to_dict()

    df_ocr["bioguide_id"]          = df_ocr["declarant_name"].map(name_to_bio)
    df_ocr["party"]                = df_ocr["declarant_name"].map(name_to_party)
    df_ocr["committee_membership"] = df_ocr["bioguide_id"].map(cmte).fillna("")
    df_ocr["committees_key_flag"]  = df_ocr["bioguide_id"].isin(key_ids)

    n_matched = df_ocr["bioguide_id"].notna().sum()
    print(f"BioGuideID résolu : {n_matched}/{len(df_ocr)} lignes ({100*n_matched/max(len(df_ocr),1):.0f}%)")
    if df_ocr["bioguide_id"].isna().any():
        unmatched = df_ocr.loc[df_ocr["bioguide_id"].isna(), "declarant_name"].unique()
        print(f"  ⚠ Non résolus : {list(unmatched)}")
    else:
        print("  ✓ Tous les membres résolus")


BioGuideID résolu : 1167/1167 lignes (100%)
  ✓ Tous les membres résolus


In [10]:
if df_ocr.empty:
    print("Aucune transaction extraite.")
else:
    print("=== Transactions par membre ===")
    summary = (
        df_ocr.groupby("declarant_name")
        .agg(n_transactions=("doc_id", "count"), n_docs=("doc_id", "nunique"))
        .reset_index()
    )
    print(summary.to_string(index=False))

    print("\n=== Répartition type de transaction ===")
    print(df_ocr["operation_type"].value_counts().to_string())

    print("\n=== Répartition fourchette de montant ===")
    print(df_ocr["amount_range"].value_counts().to_string())

    print("\n=== Répartition propriétaire ===")
    print(df_ocr["owner"].value_counts().to_string())

=== Transactions par membre ===
                declarant_name  n_transactions  n_docs
                  Adrian Smith               2       1
                    Ann Wagner               2       1
                  Brad Sherman               3       3
Charles J. "Chuck" Fleischmann              21       1
              Gus M. Bilirakis               2       1
               Keith Alan Self               5       1
             Michael T. McCaul              82       1
                  Rohit Khanna            1036       2
                     Tony Wied              11       2
              Vicente Gonzalez               3       1

=== Répartition type de transaction ===
operation_type
Purchase        641
Sale            464
Partial Sale     61
Exchange          1

=== Répartition fourchette de montant ===
amount_range
$1,001 - $15,000           939
$15,001 - $50,000          155
$50,001 - $100,000          37
$100,001 - $250,000         31
$250,001 - $500,000          3
                

In [11]:
# === Enrichissement ticker + asset_type (homogénéité avec le flux électronique) ===
# Les formulaires papier n'ont pas de colonne ticker ; on la reconstruit pour que les lignes OCR
# soient équivalentes aux lignes électroniques dans la table combinée.
#   1) ticker explicite déjà présent dans le nom OCR ("Adobe Sys Inc - ADBE", "SPDR S&P 500 ETF - SPY")
#   2) sinon, dictionnaire nom→ticker construit sur les 998 lignes électroniques déjà tickées (06)
#   3) asset_type inféré par règles calquées sur le vocabulaire électronique
# La colonne ticker_source trace l'origine (explicit / elec_dict / none) pour l'audit.

_SUFFIX_RE = re.compile(
    r"\b(CMN|COM|COMMON STOCK|COMMON|CLASS [A-Z]|CL [A-Z]|INCORPORATED|INC|CORPORATION|CORP|"
    r"COMPANY|CO|HOLDINGS|HLDGS|LLC|L\.?P\.?|LTD|PLC|THE|SYS|SYSTEMS|SER|TR|TRUST|FUND|FUNDS|ETF)\b"
)
_EXPLICIT_TICKER_RE = re.compile(r"[-(]\s*([A-Z][A-Z0-9.]{0,5})\)?\s*$")
_OCR_FIX = {"METILIFE": "METLIFE", "ATT": "AT T"}  # fautes OCR fréquentes → forme normalisée


def _norm_asset(s: str) -> str:
    s = str(s).upper()
    s = _EXPLICIT_TICKER_RE.sub("", s)            # retire un ticker en fin de chaîne
    s = re.sub(r"[^A-Z0-9 &]", " ", s)
    s = _SUFFIX_RE.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    for bad, good in _OCR_FIX.items():
        s = re.sub(rf"\b{bad}\b", good, s)
    return s


def _explicit_ticker(desc: str):
    m = _EXPLICIT_TICKER_RE.search(str(desc))
    if not m:
        return None
    t = m.group(1).strip(".")
    return t if 1 <= len(t) <= 5 else None


def _infer_asset_type(desc: str):
    d = str(desc).upper()
    if re.search(r"TREASURY|T-?BILL|T-?BOND|T-?NOTE|GOVT|GOVERNMENT|MUNICIPAL|\bMUNI\b|\bISD\b|SCHOOL DIST", d):
        return "Gov Security"
    if re.search(r"LINKED TO|NOTES? LINKED|STRUCTURED|BASKET OF", d):
        return "Other"
    if re.search(r"\bETF\b|\bFUND\b|FUNDS|INDEX|SPDR|ISHARES|VANGUARD|AMPLIFY|GLOBAL X|SELECT SECTOR|\bTR\b|TRUST", d):
        return "Mutual Fund"
    if re.search(r"\bBOND\b|\bNOTE\b|DEBENTURE|\bBILL\b", d):
        return "Corporate Bond"
    if re.search(r"CMN|COM|COMMON|\bINC\b|CORP|CLASS|\bCO\b|HOLDINGS|\bLLC\b|\bLP\b|\bPLC\b|COMPANY", d):
        return "Stock"
    return None


if df_ocr.empty:
    print("Aucune transaction — skip enrichissement ticker")
else:
    # Dictionnaire nom normalisé → ticker depuis l'électronique (06)
    elec = pd.read_csv(TABLE_DIR / "06_house_2025q1_transactions.csv", dtype=str)
    elec_tk = elec[elec["ticker"].notna() & (elec["ticker"].str.strip() != "")].copy()
    elec_tk["norm"] = elec_tk["asset_description"].map(_norm_asset)
    name_to_ticker = (
        elec_tk[elec_tk["norm"] != ""]
        .groupby("norm")["ticker"]
        .agg(lambda s: s.str.upper().mode().iat[0])
        .to_dict()
    )
    print(f"Dictionnaire nom→ticker (électronique) : {len(name_to_ticker)} entrées")

    def _resolve(desc):
        t = _explicit_ticker(desc)
        if t:
            return t.upper(), "explicit"
        t = name_to_ticker.get(_norm_asset(desc))
        if t:
            return t, "elec_dict"
        return None, "none"

    res = df_ocr["asset_description"].map(_resolve)
    df_ocr["ticker"]        = [r[0] for r in res]
    df_ocr["ticker_source"] = [r[1] for r in res]
    df_ocr["asset_type"]    = df_ocr["asset_description"].map(_infer_asset_type)

    n = len(df_ocr)
    n_tk = df_ocr["ticker"].notna().sum()
    n_at = df_ocr["asset_type"].notna().sum()
    print(f"Ticker résolu      : {n_tk}/{n} ({100*n_tk/n:.0f}%) | sources : {df_ocr['ticker_source'].value_counts().to_dict()}")
    print(f"asset_type inféré  : {n_at}/{n} ({100*n_at/n:.0f}%) | {df_ocr['asset_type'].value_counts().to_dict()}")
    print("\nÉchantillon :")
    print(df_ocr[["asset_description", "ticker", "ticker_source", "asset_type"]].head(12).to_string(index=False))

Dictionnaire nom→ticker (électronique) : 411 entrées
Ticker résolu      : 540/1167 (46%) | sources : {'none': 627, 'elec_dict': 517, 'explicit': 23}
asset_type inféré  : 1103/1167 (95%) | {'Stock': 1054, 'Mutual Fund': 30, 'Other': 14, 'Gov Security': 5}

Échantillon :
                      asset_description ticker ticker_source  asset_type
                   GE Vernova LLC (GEV)    GEV      explicit       Stock
                   GE Vernova LLC (GEV)    GEV      explicit       Stock
                  Amplify ETF Tr - HACK   HACK      explicit Mutual Fund
       Franklin Templeton ETF TR - FLJP   FLJP      explicit Mutual Fund
       Franklin Templeton ETF TR - FLJP   FLJP      explicit Mutual Fund
           Global X Funds Uranium - URA    URA      explicit Mutual Fund
          Global X Funds Defense - SHLD   SHLD      explicit Mutual Fund
               ISHARES Tr 1-5 Yr - IGSB   IGSB      explicit Mutual Fund
           ISHARES Tr 10-20 Years - TLH    TLH      explicit Mutual Fund


In [12]:
# === Passe LLM : récupération des tickers manquants (nom → ticker) ===
# Les lignes OCR sans ticker (ticker_source == "none") sont souvent de grandes valeurs cotées que
# l'électronique n'a jamais tradées (General Mills, MetLife, Colgate…) → le dictionnaire électronique
# plafonne (~46 %). On demande à Claude de mapper ces NOMS UNIQUES → ticker (action/ETF coté US ;
# null pour obligation/option/non coté). Sortie structurée forcée (tool_use) + cache versionné
# (model + prompt_sha) → un re-run sans changement ne rappelle pas l'API (gratuit).
# Quiver n'est PAS utilisé ici (seulement pour l'audit ci-dessous) : « Quiver ne sert qu'à valider ».

TICKER_LLM_CACHE = BASE_DIR / "data_v1/ticker_llm_cache.json"

_TICKER_TOOL = {
    "name": "map_tickers",
    "description": "Renvoie le symbole boursier US (ticker) pour chaque nom de titre fourni.",
    "input_schema": {
        "type": "object",
        "properties": {
            "mappings": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string", "description": "Le nom fourni, recopié à l'identique."},
                        "ticker": {"type": ["string", "null"], "description": "Ticker US en MAJUSCULES (ex. GIS, MET, AAPL). null si ce n'est pas une action/ETF cotée en bourse US."},
                        "is_equity": {"type": "boolean", "description": "true si action ordinaire ou ETF coté US."},
                    },
                    "required": ["name", "ticker", "is_equity"],
                },
            }
        },
        "required": ["mappings"],
    },
}
_TICKER_PROMPT = (
    "Tu reçois des noms de titres financiers issus de déclarations de transactions du Congrès US "
    "(formulaires PTR). Pour CHAQUE nom, donne le ticker boursier US s'il s'agit d'une action "
    "ordinaire ou d'un ETF coté aux États-Unis. Mets ticker=null si c'est une obligation, une option, "
    "un bon du Trésor, un fonds non coté, un titre privé, ou si tu n'es pas sûr. Ne devine jamais un "
    "ticker au hasard : dans le doute, null. Recopie 'name' à l'identique.\n\nNoms :\n{names}"
)
_TICKER_PROMPT_SHA = hashlib.sha256(
    (_TICKER_PROMPT + json.dumps(_TICKER_TOOL, sort_keys=True) + MODEL).encode()
).hexdigest()[:16]
_VALID_TICKER = re.compile(r"^[A-Z][A-Z.]{0,5}$")


def _load_ticker_cache():
    if TICKER_LLM_CACHE.exists():
        obj = json.loads(TICKER_LLM_CACHE.read_text(encoding="utf-8"))
        if obj.get("prompt_sha") == _TICKER_PROMPT_SHA and obj.get("model") == MODEL:
            return obj.get("mappings", {})
    return {}


def _save_ticker_cache(mappings):
    TICKER_LLM_CACHE.write_text(
        json.dumps({"model": MODEL, "prompt_sha": _TICKER_PROMPT_SHA, "mappings": mappings},
                   ensure_ascii=False, indent=1), encoding="utf-8")


def _map_tickers_batch(cli, names, max_retries=4):
    listing = "\n".join(f"- {n}" for n in names)
    last = None
    for attempt in range(max_retries):
        try:
            resp = cli.messages.create(
                model=MODEL, max_tokens=MAX_TOKENS,
                tools=[_TICKER_TOOL], tool_choice={"type": "tool", "name": "map_tickers"},
                messages=[{"role": "user", "content": _TICKER_PROMPT.format(names=listing)}],
            )
            for block in resp.content:
                if getattr(block, "type", None) == "tool_use" and block.name == "map_tickers":
                    return block.input.get("mappings", [])
            return []
        except (anthropic.RateLimitError, anthropic.APIStatusError, anthropic.APIConnectionError) as e:
            last = e
            if attempt < max_retries - 1:
                time.sleep(min(2 ** attempt, 20)); continue
            raise
    raise RuntimeError(f"map_tickers échec : {last}")


if df_ocr.empty:
    print("Aucune transaction — skip passe LLM ticker")
else:
    _tkcli = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    mask_none = df_ocr["ticker_source"].eq("none")
    missing_names = sorted(df_ocr.loc[mask_none, "asset_description"].dropna().unique())
    print(f"Noms uniques sans ticker : {len(missing_names)} (sur {int(mask_none.sum())} lignes)")

    cache = _load_ticker_cache()
    todo = [n for n in missing_names if n not in cache]
    print(f"Cache LLM : {len(missing_names) - len(todo)} hits | à résoudre via API : {len(todo)}")

    BATCH = 60
    for i in range(0, len(todo), BATCH):
        batch = todo[i:i + BATCH]
        for m in _map_tickers_batch(_tkcli, batch):
            nm = m.get("name", "")
            tk = (m.get("ticker") or "").upper().strip(".")
            cache[nm] = tk if (m.get("is_equity") and _VALID_TICKER.match(tk)) else ""
        for n in batch:                       # noms non renvoyés → vides (évite de redemander)
            cache.setdefault(n, "")
        print(f"  batch {i // BATCH + 1} : {min(i + BATCH, len(todo))}/{len(todo)}")
    _save_ticker_cache(cache)

    # Application : remplir les lignes "none" dont le nom a un ticker LLM
    _fill = df_ocr["ticker_source"].eq("none") & df_ocr["asset_description"].map(
        lambda d: bool(cache.get(d, "")))
    df_ocr.loc[_fill, "ticker"] = df_ocr.loc[_fill, "asset_description"].map(lambda d: cache.get(d, ""))
    df_ocr.loc[_fill, "ticker_source"] = "llm"

    n = len(df_ocr)
    n_tk = int((df_ocr["ticker"].fillna("").astype(str).str.strip() != "").sum())
    print(f"\nTicker après passe LLM : {n_tk}/{n} ({100 * n_tk / n:.0f}%)  [avant LLM : 540/1167 = 46%]")
    print(f"Sources : {df_ocr['ticker_source'].value_counts().to_dict()}")


Noms uniques sans ticker : 310 (sur 627 lignes)
Cache LLM : 310 hits | à résoudre via API : 0

Ticker après passe LLM : 1055/1167 (90%)  [avant LLM : 540/1167 = 46%]
Sources : {'elec_dict': 517, 'llm': 515, 'none': 112, 'explicit': 23}


In [13]:
# === Audit : précision des tickers LLM vs QuiverQuant (Quiver = arbitre, jamais une source) ===
# On rapproche nos lignes 'llm' du couple (déposant, nom normalisé) de Quiver, qui porte ticker +
# company. Concordance attendue élevée ; les écarts sont souvent des artefacts Quiver (tickers de
# parts préférentielles « -PA/-PG », ou société voisine) — on les liste pour transparence.
if not df_ocr.empty and (df_ocr["ticker_source"] == "llm").any():
    _qa = pd.read_csv(BASE_DIR / "data_v1/external/quiver_congress_trading_2025.csv", dtype=str)
    _qa = _qa[_qa["chamber"].str.lower() == "house"].copy()

    _AUD_SUF = re.compile(r"\b(CMN|COM|COMMON STOCK|COMMON|CLASS [A-Z]|CL [A-Z]|INCORPORATED|INC|"
                          r"CORPORATION|CORP|COMPANY|CO|HOLDINGS|HLDGS|LLC|L\.?P\.?|LTD|PLC|THE|SYS|"
                          r"SYSTEMS|GROUP|GRP)\b")

    def _audnorm(s):
        s = re.sub(r"\([^)]*\)", " ", str(s).upper())
        s = re.sub(r"[^A-Z0-9 &]", " ", s)
        s = _AUD_SUF.sub(" ", s)
        return re.sub(r"\s+", " ", s).strip()

    _qa["norm"] = _qa["company"].map(_audnorm)
    _qmap = (_qa[_qa["ticker"].notna() & (_qa["ticker"].str.strip() != "")]
             .groupby(["bioguide_id", "norm"])["ticker"]
             .agg(lambda s: s.str.upper().mode().iat[0]).to_dict())

    _rows = []
    for _, r in df_ocr[df_ocr["ticker_source"] == "llm"].iterrows():
        qt = _qmap.get((r.get("bioguide_id"), _audnorm(r["asset_description"])))
        if qt:
            _rows.append({"declarant_name": r["declarant_name"], "asset_description": r["asset_description"],
                          "ticker_llm": r["ticker"], "ticker_quiver": qt, "match": r["ticker"] == qt})
    audit = pd.DataFrame(_rows, columns=["declarant_name", "asset_description", "ticker_llm", "ticker_quiver", "match"])
    audit.to_csv(TABLE_DIR / "06e_ticker_llm_audit.csv", index=False)

    n_llm = int((df_ocr["ticker_source"] == "llm").sum())
    if len(audit):
        ok = int(audit["match"].sum())
        print(f"Audit LLM vs Quiver : {ok}/{len(audit)} concordants ({100 * ok / len(audit):.0f}%) "
              f"— {n_llm} tickers LLM au total, {len(audit)} rapprochables de Quiver.")
        print("→ 06e_ticker_llm_audit.csv écrit.")
        disc = audit[~audit["match"]]
        if len(disc):
            print(f"Écarts ({len(disc)}) — souvent artefacts Quiver (parts préf. -PA/-PG, société voisine) :")
            print(disc[["asset_description", "ticker_llm", "ticker_quiver"]].head(8).to_string(index=False))
    else:
        print(f"Audit : {n_llm} tickers LLM, aucun rapprochable de Quiver.")
else:
    print("Pas de ticker LLM — audit ignoré.")


Audit LLM vs Quiver : 466/500 concordants (93%) — 515 tickers LLM au total, 500 rapprochables de Quiver.
→ 06e_ticker_llm_audit.csv écrit.
Écarts (34) — souvent artefacts Quiver (parts préf. -PA/-PG, société voisine) :
                 asset_description ticker_llm ticker_quiver
  APOLLO GLOBAL MANAGEMENT INC CMN        APO        AAM-PA
                EATON CORP PLC CMN        ETN           ELN
     AMERICAN INTL GROUP, INC. CMN        AIG           AFG
      W.R. BERKLEY CORPORATION CMN        WRB        WRB-PG
VERTEX PHARMACEUTICALS INCORPO CMN       VRTX          VERX
VERTEX PHARMACEUTICALS INCORPO CMN       VRTX          VERX
                EATON CORP PLC CMN        ETN           ELN
     AMERICAN INTL GROUP, INC. CMN        AIG           AFG


## 8. Sauvegarde et merge avec la table principale

- `06b_house_2025q1_ocr_transactions.csv` : transactions OCR seules
- `06_house_2025q1_transactions_FINAL.csv` : table complète (digital + OCR)

In [14]:
# Colonnes dans l'ordre exact de la table principale (+ ticker_source, provenance en extra)
import sys
sys.path.insert(0, str(BASE_DIR))
import sector_enrich as se   # enrichissement secteur GICS -> ETF SPDR (module local, voir sector_enrich.py)

SCHEMA_COLS = [
    "bioguide_id", "declarant_name", "chamber", "party", "state_district",
    "committee_membership", "committees_key_flag", "transaction_date", "disclosure_date",
    "ticker", "asset_description", "asset_type", "operation_type",
    "amount_range", "amount_midpoint", "amount_split_flag",
    "owner", "doc_id", "source_url", "natural_key_hash",
]
EXTRA_COLS = ["ticker_source", "provenance"]

df_ocr_out = df_ocr.reindex(columns=SCHEMA_COLS + [c for c in EXTRA_COLS if c in df_ocr.columns])

# Sauvegarde OCR seule
ocr_path = TABLE_DIR / "06b_house_2025q1_ocr_transactions.csv"
df_ocr_out.to_csv(ocr_path, index=False)
print(f"-> {ocr_path.name} : {len(df_ocr_out)} lignes")

# Merge avec la table principale (electronique)
main_path = TABLE_DIR / "06_house_2025q1_transactions.csv"
if main_path.exists():
    df_main = pd.read_csv(main_path, dtype={"doc_id": str})
    df_main["provenance"] = "house-pdf-electronic"
    print(f"Table principale : {len(df_main)} lignes")
else:
    df_main = pd.DataFrame(columns=SCHEMA_COLS + ["provenance"])
    print("Table principale introuvable - la table v2 contiendra uniquement les donnees OCR")

# DEDUPLICATION INTER-SOURCES UNIQUEMENT.
# On retire une ligne OCR seulement si elle est strictement identique a une ligne electronique
# (collision reelle entre les deux flux). On ne deduplique JAMAIS au sein de l'OCR : les lignes
# repetees a l'identique sont des transactions reelles distinctes (verifie sur les formulaires
# Khanna - ex. COLGATE x2, GENERAL MILLS x3 le meme jour). Dedupliquer intra-OCR detruisait de la donnee.
elec_hashes = set(df_main["natural_key_hash"].dropna())
collide = df_ocr_out["natural_key_hash"].isin(elec_hashes)
n_collide = int(collide.sum())
df_combined = pd.concat([df_main, df_ocr_out[~collide]], ignore_index=True)
if n_collide:
    print(f"  {n_collide} ligne(s) OCR retiree(s) car identiques a une ligne electronique")

# === Enrichissement secteur GICS -> ETF SPDR (champs obligatoires sector_gics, etf_proxy) ===
# Jointure sur `ticker` (deja stabilise) : yfinance (factuel) + repli LLM, cache versionne.
# Purement additif - n'altere aucune colonne existante. Detail : sector_enrich.py.
df_combined = se.enrich_sectors(df_combined)

# Ordre final : sector_gics/etf_proxy regroupes avec les champs obligatoires ; tracabilite en fin.
FINAL_COLS = [
    "bioguide_id", "declarant_name", "chamber", "party", "state_district",
    "committee_membership", "committees_key_flag", "transaction_date", "disclosure_date",
    "ticker", "asset_description", "asset_type", "sector_gics", "etf_proxy",
    "operation_type", "amount_range", "amount_midpoint", "amount_split_flag",
    "owner", "doc_id", "source_url", "natural_key_hash",
    "ticker_source", "sector_source", "provenance",
]
df_combined = df_combined.reindex(columns=FINAL_COLS + [c for c in df_combined.columns if c not in FINAL_COLS])

final_path = TABLE_DIR / "06_house_2025q1_transactions_FINAL.csv"
df_combined.to_csv(final_path, index=False)

print(f"\n-> {final_path.name} : {len(df_combined)} lignes ({len(df_main)} electronique + {len(df_ocr_out) - n_collide} OCR)")
print(f"   coverage : {117 - len(scanned)} electronique + {len(scanned)} OCR = 117/117 PTRs")
print(f"   colonnes : {len(df_combined.columns)} (dont sector_gics + etf_proxy)")
print("\nRepartition provenance :")
print(df_combined["provenance"].value_counts().to_string())

-> 06b_house_2025q1_ocr_transactions.csv : 1167 lignes
Table principale : 1105 lignes
Tickers distincts valides : 591 | cache : 591 hits | à résoudre : 0

-> 06_house_2025q1_transactions_FINAL.csv : 2272 lignes (1105 electronique + 1167 OCR)
   coverage : 100 electronique + 17 OCR = 117/117 PTRs
   colonnes : 25 (dont sector_gics + etf_proxy)

Repartition provenance :
provenance
house-pdf-ocr           1167
house-pdf-electronic    1105


In [15]:
# === Audit secteur : couverture + concordance croisee yfinance <-> LLM (table 06f) ===
# Meme esprit que l'audit ticker 06e : on chiffre la fiabilite de l'enrichissement.
sector_audit = se.build_audit(df_combined)
sector_audit.to_csv(TABLE_DIR / "06f_sector_audit.csv", index=False)
se.summary(sector_audit)

n = len(df_combined)
cov = int(df_combined["sector_gics"].notna().sum())
print(f"\nsector_gics rempli : {cov}/{n} lignes = {cov / n:.1%}")
print(f"etf_proxy rempli   : {int(df_combined['etf_proxy'].notna().sum())}/{n}")
print(f"sources            : {df_combined['sector_source'].value_counts().to_dict()}")


=== Audit secteur (591 tickers distincts) ===
  sources : {'yfinance': 548, 'llm': 32, 'none': 11}
  audit croisé yfinance↔LLM : 543 tickers comparables, accord = 95.4%
  désaccords (25) — extrait :
ticker                     yf                    llm
   APG            Industrials              Materials
   APP Communication Services Information Technology
   AVY Consumer Discretionary              Materials
  BALL Consumer Discretionary              Materials
   CCK Consumer Discretionary              Materials
 CLPBY            Health Care       Consumer Staples
  CPAY Information Technology             Financials
    DG       Consumer Staples Consumer Discretionary
   FIS Information Technology             Financials
   FTV Information Technology            Industrials
 GBOOY             Financials       Consumer Staples
   GPK Consumer Discretionary              Materials
   GPN            Industrials             Financials
  HLMN            Industrials       Consumer Staples
   IB

In [16]:
# === QA — contrôles de cohérence (signalés bruyamment + persistés) ===
qa = df_ocr_out.copy()
qa["td"] = pd.to_datetime(qa["transaction_date"], errors="coerce")

leak      = qa[qa["asset_description"].str.contains("example|mega corp", case=False, na=False)]
oob       = qa[(qa["td"] < "2025-01-01") | (qa["td"] > "2025-03-31")]
bad_op    = qa[~qa["operation_type"].isin(["Purchase", "Sale", "Partial Sale", "Exchange"])]
empty_amt = qa[qa["amount_range"].fillna("") == ""]

checks = [("ligne_exemple_non_filtree", leak), ("date_hors_Q1", oob),
          ("operation_type_hors_enum", bad_op), ("amount_range_vide", empty_amt)]

print("=== QA OCR ===")
issues = [(name, len(df_)) for name, df_ in checks if len(df_)]
if issues:
    for name, n in issues:
        print(f"  ✗ {name} : {n}")
    print("⚠ Anomalies — voir 06c_ocr_qa_flags.csv")
else:
    print("  ✓ aucune anomalie : pas de ligne-exemple, dates dans Q1, enums valides, montants présents.")

parts = [df_.assign(flag=name)[["doc_id", "declarant_name", "flag", "transaction_date", "asset_description"]]
         for name, df_ in checks if len(df_)]
qa_flags = pd.concat(parts, ignore_index=True) if parts else \
    pd.DataFrame(columns=["doc_id", "declarant_name", "flag", "transaction_date", "asset_description"])
qa_flags.to_csv(TABLE_DIR / "06c_ocr_qa_flags.csv", index=False)

=== QA OCR ===
  ✗ amount_range_vide : 1
⚠ Anomalies — voir 06c_ocr_qa_flags.csv


In [17]:
# === Validation externe vs QuiverQuant (comptage par déposant) ===
# Fenêtre COMPARABLE : nos PTR sont DIVULGUÉS en Q1, donc on filtre Quiver sur la DATE DE
# DIVULGATION en Q1 (pas la date de transaction — sinon on compte les trades de mars divulgués
# en avril, dans un PTR hors de notre périmètre Q1, ce qui crée un faux « sous-comptage »).
# Quiver SOUS-recense les dépôts papier denses → un OCR ≥ Quiver est attendu et sain
# (vérifié visuellement : McCaul = 82 lignes OCR ≈ 4 pages × ~20 lignes du formulaire).

# CSV Quiver embarqué localement (copie dans data_v1/external/) → aucun accès hors dossier.
QUIVER_CSV = BASE_DIR / "data_v1/external/quiver_congress_trading_2025.csv"

if QUIVER_CSV.exists() and not df_ocr.empty:
    q = pd.read_csv(QUIVER_CSV, dtype=str)
    q = q[q["chamber"].str.lower() == "house"].copy()
    q["dd"] = pd.to_datetime(q["disclosure_date"], errors="coerce")
    qD = q[(q["dd"] >= "2025-01-01") & (q["dd"] <= "2025-03-31")]

    cmp = (df_ocr_out.groupby("bioguide_id")
           .agg(name=("declarant_name", "first"), docs=("doc_id", "nunique"), n_ocr=("doc_id", "count"))
           .reset_index())
    cmp["n_quiver_discQ1"] = cmp["bioguide_id"].map(qD.groupby("bioguide_id").size()).fillna(0).astype(int)
    cmp["delta"] = cmp["n_ocr"] - cmp["n_quiver_discQ1"]
    cmp["verdict"] = cmp.apply(
        lambda r: "quiver_sans_donnee" if r["n_quiver_discQ1"] == 0
        else ("ocr>=quiver" if r["delta"] >= 0
              else ("concordant" if abs(r["delta"]) <= max(3, 0.1 * r["n_quiver_discQ1"]) else "ocr_souscompte")),
        axis=1,
    )
    cmp = cmp.sort_values("n_ocr", ascending=False)
    cmp.to_csv(TABLE_DIR / "06d_ocr_quiver_comparison.csv", index=False)

    print("=== OCR vs QuiverQuant (PTR divulgués en Q1 2025, par déposant) ===")
    print(cmp.to_string(index=False))
    print(f"\nSource Quiver : {QUIVER_CSV.relative_to(BASE_DIR)}")
    print("→ 06d écrit. OCR ≥ Quiver = OCR au moins aussi complet (Quiver sous-recense les dépôts papier denses).")
    print("  Quiver=0 = lacune Quiver sur petit dépôt papier (pas une erreur OCR, confirmé visuellement).")
else:
    print("Quiver introuvable ou OCR vide — validation par comptage ignorée")


=== OCR vs QuiverQuant (PTR divulgués en Q1 2025, par déposant) ===
bioguide_id                           name  docs  n_ocr  n_quiver_discQ1  delta            verdict
    K000389                   Rohit Khanna     2   1036              756    280        ocr>=quiver
    M001157              Michael T. McCaul     1     82               56     26        ocr>=quiver
    F000459 Charles J. "Chuck" Fleischmann     1     21               13      8        ocr>=quiver
    W000829                      Tony Wied     2     11                4      7        ocr>=quiver
    S001224                Keith Alan Self     1      5                0      5 quiver_sans_donnee
    G000581               Vicente Gonzalez     1      3                0      3 quiver_sans_donnee
    S000344                   Brad Sherman     3      3                0      3 quiver_sans_donnee
    B001257               Gus M. Bilirakis     1      2                0      2 quiver_sans_donnee
    S001172                   Adrian Smit

In [18]:
# === Packaging / rendu final : un classeur Excel unique et complet ===
from pathlib import Path
import openpyxl  # noqa  (moteur xlsx)

final_csv = TABLE_DIR / "06_house_2025q1_transactions_FINAL.csv"
xlsx_out  = TABLE_DIR / "house_2025q1_FINAL.xlsx"

def _safe_csv(p, **kw):
    try:
        df = pd.read_csv(p, **kw)
        return df if len(df.columns) else pd.DataFrame()
    except Exception:
        return pd.DataFrame()

tx  = pd.read_csv(final_csv, dtype=str)                      # table FINALE (2272)
tx["_is_ocr"] = (tx["provenance"] == "house-pdf-ocr").astype(int)
syn = (tx.groupby(["declarant_name", "party", "state_district"], dropna=False)
         .agg(n_transactions=("doc_id", "count"), n_ocr=("_is_ocr", "sum"))
         .reset_index())
syn["n_electronique"] = syn["n_transactions"] - syn["n_ocr"]
syn = syn.sort_values("n_transactions", ascending=False)
tx = tx.drop(columns="_is_ocr")

val = _safe_csv(TABLE_DIR / "06d_ocr_quiver_comparison.csv")
qaf = _safe_csv(TABLE_DIR / "06c_ocr_qa_flags.csv")
ptr = _safe_csv(TABLE_DIR / "03_ptr_index.csv", dtype=str)

lisez = pd.DataFrame({
    "onglet": ["transactions", "synthese_membres", "validation_quiver", "qa_flags", "ptr_index"],
    "contenu": [
        f"Table COMPLETE House Q1 2025 ({len(tx)} lignes = electronique + OCR). LE rendu final.",
        "Nombre de transactions par membre, ventile electronique / OCR.",
        "Comparaison OCR vs QuiverQuant (recall ~95%, OCR >= Quiver ; voir RAPPORT_OCR_VALIDATION.md).",
        "Anomalies de coherence detectees par la QA (vide ou quasi vide = bon signe).",
        "Index des 117 PTR (depots) de la fenetre Q1 2025.",
    ],
})

with pd.ExcelWriter(xlsx_out, engine="openpyxl") as w:
    lisez.to_excel(w, sheet_name="LISEZMOI", index=False)
    tx.to_excel(w, sheet_name="transactions", index=False)
    syn.to_excel(w, sheet_name="synthese_membres", index=False)
    if len(val): val.to_excel(w, sheet_name="validation_quiver", index=False)
    if len(qaf): qaf.to_excel(w, sheet_name="qa_flags", index=False)
    if len(ptr): ptr.to_excel(w, sheet_name="ptr_index", index=False)

print(f"-> {xlsx_out.name} ecrit : {len(tx)} transactions | feuilles : LISEZMOI, transactions, "
      f"synthese_membres, validation_quiver, qa_flags, ptr_index")


-> house_2025q1_FINAL.xlsx ecrit : 2272 transactions | feuilles : LISEZMOI, transactions, synthese_membres, validation_quiver, qa_flags, ptr_index


---
## Notes techniques (v2 — fiabilisée)

**Robustesse de l'extraction :**
- **Sortie structurée forcée** (`tool_use` sur `record_transactions`) : plus de JSON-dans-le-texte ni de troncature à réparer ; `transaction_type`, `owner`, `amount_code` validés par enum à la source.
- **Échecs jamais silencieux** : tout batch en erreur est journalisé dans `06c_ocr_failures.csv` (un fichier vide = succès complet). Retry/backoff sur 429/5xx.
- **Cache versionné** : `ocr_cache/{doc_id}.json` est un objet auditable `{model, prompt_sha, batches[…], status, transactions}`. Il est réutilisé seulement si `model` + `prompt_sha` correspondent — modifier le prompt invalide automatiquement le cache. Le `status` distingue `nothing_to_report` d'un `partial_error`.

**Pièges du formulaire papier traités :**
- **Ligne d'exemple pré-imprimée** « Example: Mega Corp. Common Stock » : explicitement exclue par le prompt (était la source des fausses transactions).
- **Répétitions réelles** (même actif/date/montant le même jour) : conservées. La déduplication n'opère qu'**entre** sources (électronique vs OCR), jamais au sein de l'OCR — sinon on détruit des transactions réelles.
- **« Nothing to report for <mois> »** : correctement renvoyé vide (ex. Hal Rogers jan./fév.).

**Enrichissement (homogénéité avec l'électronique) :**
- `ticker` reconstruit : ticker explicite dans le nom (« … - ADBE ») puis dictionnaire nom→ticker bâti sur les 998 lignes électroniques tickées ; `ticker_source` trace l'origine.
- `asset_type` inféré par règles (Gov Security / Mutual Fund / Stock / Corporate Bond / Other).

**Validation :** comparaison de comptage par déposant vs QuiverQuant → `06d_ocr_quiver_comparison.csv`. Quiver couvre les gros déposants (Khanna, McCaul) mais a des lacunes sur les petits dépôts papier — un `quiver_sans_donnee` n'est donc pas une erreur OCR (confirmé visuellement, PDF source vs CSV).

**Passage au corpus complet (2013–2025, ~2 431 PDFs) :** le notebook est réutilisable tel quel sur tout `PDF_DIR`. Budget ≈ 2 431 × 3 pages × $0.006 ≈ **$44**. Le `06c_ocr_failures.csv` sert de tracker de statut ; relancer ne re-traite que les PDFs dont le cache est absent ou périmé.